# CS 3120/5120: Secure Distributed Computation
## Homework 7

In [ ]:
# Useful imports and utility functions
import pychor
import numpy as np
import galois
import random
bits = 128

from collections import defaultdict
from collections import namedtuple

## Definitions: Paillier Cryptosystem

In [ ]:
@pychor.local_function
def keygen(bits):
    """Generates keys with `bits`-bits of security. Returns a pair: (secret key, public key)."""
    def invmod(x, m):
        gcd, s, t = galois.egcd(x, m)
        assert gcd == 1
        return s

    p = galois.random_prime(int(bits/2))
    q = galois.random_prime(int(bits/2))

    n = p*q
    g = n+1
    lamb = (p-1) * (q-1)
    mu = invmod(lamb, n)
    
    sk = (lamb, mu)
    pk = (n, g)
    return sk, pk

@pychor.local_function
def encrypt(m, pk):
    """Encrypts the message `m` with public key `pk`."""
    n, g = pk
    n_sq = n**2
    r = random.randint(1, n)
    c = (pow(g, m, n_sq) * pow(r, n, n_sq)) % n_sq
    return c

@pychor.local_function
def decrypt(c, sk, pk):
    """Decrypts the ciphertext `c` using secret key `sk` and public key `pk`."""
    lamb, mu = sk
    n, g = pk
    n_sq = n**2
    L_result = (pow(c, lamb, n_sq) - 1)//n
    return (L_result * mu) % n

@pychor.local_function
def e_add(c1, c2, pk):
    """Add one encrypted integer to another"""
    n, g = pk

    return c1 * c2 % n**2

## Question 1 (30 points)

Implement an electronic voting system using the Paillier cryptosystem. The system has two parts: a voter encrypts their vote using the cryptosystem and submits it to the server, and the server stores the encrypted votes and tallies them at the end of the election. The voting server holds the public and private keys; the public key is available to the voters, so that they can encrypt their votes. Generate your public and private keys with at least 32-bit security.

Your `run_election` function should implement a protocol where the server distributes the public key to the voters, each voter submits an encrypted vote, the server tallies the votes, the server decrypts the totals, and the server returns a dictionary mapping each candidate to the number of votes they received.

In [ ]:
candidates = ['Candidate 1',
              'Candidate 2',
              'Candidate 3']

voters = [pychor.Party(f'voter{i}') for i in range(100)]
voting_server = pychor.Party('voting_server')

In [ ]:
def run_election(candidates, voting_server, voters, votes):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
with pychor.LocalBackend():
    votes = {voter: voter.constant(np.random.choice(candidates)) for voter in voters}
    expected_results = {c: np.sum(np.array([v.val for v in votes.values()]) == c) for c in candidates}
    print('Expected election results:', expected_results)

    results = run_election(candidates, voting_server, voters, votes)
    print('Election results:', results)

    assert {c: r.val for c, r in results.items()} == expected_results

## Question 2 (10 points)

In 2-5 sentences, answer the following:

1. What trust assumptions do we make about the *voting server* in this election system?
2. What trust assumptions do we make about the *voter* in this election system?

YOUR ANSWER HERE

## Question 3 (10 points)

In 2-5 sentences, answer the following:

1. What is one way a malicious *voting server* could break the rules of the election?
2. What are two ways a malicious *voter* could break the rules of the election?

YOUR ANSWER HERE

## Question 4 (30 points)

The [Boston Women's Workforce Council Gender Pay Gap Survey](https://thebwwc.org/mpc) uses MPC to deploy a survey of Boston-area businesses to determine how women and men are paid differently. Each business submits encrypted values for their employees' salaries, and the system calculates the average salaries for women and men across all of the businesses. This design protects the privacy of individual employees, and protects individual businesses from embarrassment.

Implement a system for conducting a survey like this using the Paillier cryptosystem. Participants should submit their own salaries, and specify their gender as either male or female. The survey server should collect responses, and at the end of the survey, calculate the *average salaries for men and women*. Use at least 32-bit security. Your approach may reveal the number of women and men participating, in addition to the average salaries.

In [ ]:
survey_participants = [pychor.Party(f'participant{i}') for i in range(100)]
survey_server = pychor.Party('survey_server')

In [ ]:
def run_survey(survey_server, survey_participants, responses):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
def random_response():
    gender = np.random.choice(['male', 'female'])
    salary = np.random.randint(10000, 100000)
    return (gender, salary)

with pychor.LocalBackend():
    all_responses = [random_response() for _ in survey_participants]
    expected_male_salary = np.array([r[1] for r in all_responses])[[r[0] == 'male' for r in all_responses]].mean()
    expected_female_salary = np.array([r[1] for r in all_responses])[[r[0] == 'female' for r in all_responses]].mean()
    print('Average expected male salary:', expected_male_salary)
    print('Average expected female salary:', expected_female_salary)
    
    responses = {participant: participant.constant(r) for participant, r in zip(survey_participants, all_responses)}

    avg_female_salary, avg_male_salary = run_survey(survey_server, survey_participants, responses)
    print('Average male salary:', avg_male_salary)
    print('Average female salary:', avg_female_salary)

    assert avg_female_salary.val == expected_female_salary
    assert avg_male_salary.val == expected_male_salary

## Question 5 (10 points)

In 2-5 sentences, answer the following:

1. What is one way a malicious *survey server* could break the rules of the survey?
2. What are two ways a malicious *survey participant* could break the rules of the survey?

YOUR ANSWER HERE

## Question 6 (10 points)

In 2-5 sentences each, answer the following:

1. How are the approaches in this assignment (based on partially homomorphic encryption) **better** than alternative MPC protocols we could have used? Why?
2. How are the approaches in this assignment (based on partially homomorphic encryption) **worse** than alternative MPC protocols we could have used? Why?

YOUR ANSWER HERE